# Attitude + Conical Sensor — 2D RT Workflow

Extends `workflow_2d_rt.ipynb` with two new capabilities:

1. **Rigid-body attitude dynamics** — each vehicle carries a quaternion
   attitude state and body-frame angular rates, integrated via RK4 with
   per-axis reaction-wheel saturation.  The guard's `ATTITUDE_CONTROL`
   action component lets it command body-frame torques each step; the
   bandit precesses freely (zero torque, free rotation).

2. **Body-frame conical sensor** — instead of the omnidirectional
   `RangeLimitedObservation`, the guard uses `ConicalObservation`, which
   only generates a measurement when the target's line-of-sight vector
   falls inside at least one of the guard's body-fixed sensor cones.
   The cones rotate with the vehicle, so *pointing* becomes a
   decision-relevant action.

**The core tradeoff** the conical model introduces: the agent can spin
rapidly to scan a wide field (cheap in torque, narrow window per target)
or stay pointed at a high-priority target (expensive in opportunity cost
but steady measurements, allowing the EKF to converge quickly). This
tradeoff is invisible in the omnidirectional model.

**Compared to `workflow_2d_rt.ipynb`**, the state grows by 7 per vehicle:
4 (quaternion) + 3 (body rates) + 3 (applied torque transient) = 10 extra
fields beyond the RT+MASS base.

**Setup** — run with `uv`:
```bash
JAX_PLATFORMS=cpu uv run --with jupyter --with ipykernel jupyter notebook examples/workflow_2d_rt_attitude.ipynb
```


## 0. Device setup (Apple Silicon safety guard)

`orbitalgym` defaults to CPU + float32 at import so it works on every JAX backend (CPU, CUDA, MPS). For full orbit precision call `orbitalgym.set_precision(jnp.float64)` after import — this enables `jax_enable_x64`, which Apple's MLX backend rejects, so if `jax-mps` is installed we also pin `JAX_PLATFORMS=cpu` here as a defensive layer before importing `jax`. See `examples/running_on_mps.ipynb` for the full story.

In [1]:
import importlib.metadata
import os

try:
    importlib.metadata.distribution('jax-mps')
    os.environ.setdefault('JAX_PLATFORMS', 'cpu,mps')
    os.environ.setdefault('JAX_DEFAULT_DEVICE', 'cpu')
except importlib.metadata.PackageNotFoundError:
    os.environ.setdefault('JAX_PLATFORMS', 'cpu')

In [2]:
# Auto-reload: picks up changes to orbitalgym without a kernel restart.
%load_ext autoreload
%autoreload 2

import pathlib
import shutil
import tempfile
from types import SimpleNamespace

import jax
import jax.numpy as jnp
import matplotlib
matplotlib.use("Agg")  # non-interactive backend for notebook execution
import matplotlib.pyplot as plt
import matplotlib.animation as mpl_animation

from orbitalgym import (
    OrbitalGymEnv,
    ScenarioConfig,
    VehicleParamsSpec,
    rollout,
    save_run,
    load_run,
)
from orbitalgym.belief import EKFBeliefUpdater, EKFFromTruthInitializer
from orbitalgym.dynamics.attitude import AttitudeParams
from orbitalgym.dynamics.hcw import hcw_rt_step
from orbitalgym.env.types import Actions, BySide, Side
from orbitalgym.observations.conical import ConicalObservation
from orbitalgym.policies import ZeroControl
from orbitalgym.reference_orbit import ReferenceOrbitState, mean_motion as ref_mean_motion
from orbitalgym.registry import (
    ActionComponentKey,
    AttitudeDynamicsKey,
    DynamicsKey,
    Frame,
    StateComponentKey,
)
from orbitalgym.sampling.attitude import FixedAttitude
from orbitalgym.sampling.mass import ConstantMass
from orbitalgym.sampling.side import RelativeEllipse
from orbitalgym.sampling.spec import ICSpec
from orbitalgym.viz import (
    RolloutScene,
    interactive_viewer,
    plot_reward_curve,
    plot_rollout_panels,
    save_animation,
)
from orbitalgym.viz import plot_rt

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Platform 'mps' is experimental and not all JAX functionality may be correctly supported!


## 1. Scenario configuration — attitude + conical sensor

Three additions beyond the standard 2D RT config:

### Why ATTITUDE + BODY_RATES components?
`StateComponentKey.ATTITUDE` adds a (w, x, y, z) quaternion; `BODY_RATES`
adds angular velocity (ωx, ωy, ωz) in the body frame.  Together they give
the state enough structure for `ConicalObservation` to rotate the body-fixed
boresight into the world frame each step.  `APPLIED_TORQUE` is auto-added
by the framework when `ATTITUDE_CONTROL` is in the action components — it
is a transient that carries the commanded torque through the action→dynamics
handoff inside `env.step`.

### Why ATTITUDE_CONTROL action component?
`ActionComponentKey.IMPULSIVE_MANEUVER` gives translational authority (Δv).
`ActionComponentKey.ATTITUDE_CONTROL` gives rotational authority (body-frame
torque).  In 2D, the torque field is 1-dimensional (only body-z, which
rotates the spacecraft in the orbital plane) — the component lifts it to
a full 3-vector internally before passing it to `rigid_body_attitude_step`.

### Why AttitudeParams?
`AttitudeParams` carries two physical knobs:
- `inertia_diag` — principal-axis inertia tensor (kg m²), used by Euler's
  equation to convert torque → angular acceleration.
- `omega_max` — per-axis saturation limit (rad/s), modeling reaction-wheel
  maximum speed.  In 2D we saturate only the z-axis.

### Why the proto-then-final pattern?
`ConicalObservation` needs the `StateLayout`, which is determined entirely
by the vehicle composition (n_guards, n_bandits, components).  We build a
lightweight proto env just to get the layout, construct the observation
function, then build the final `ScenarioConfig` with it wired in.


### AttitudeSampler family

The new `AttitudeSampler` family (`IdentityAttitude`, `FixedAttitude`, `UniformAttitude`, `UniformBodyRates`, `UniformAttitudeAndRates`) plugs into `RelativeKeplerian` and `RelativeEllipse` via their `attitude_sampler` field. We use `FixedAttitude` here for a deterministic slow spin (0.1°/s about z) so the cones sweep visibly across the rollout. Switch to `UniformBodyRates(omega_max_rad_s=(...))` for randomized initial spin or `UniformAttitude()` for randomized initial orientation.

In [ ]:
# ── Sensor boresight knob ──────────────────────────────────────────────────
# Four-face configuration: +R, -R, +T, -T boresights in body frame.
# In 2D, the spacecraft rotates only about z, so the relevant boresights are
# in the x-y body plane (body-z = N-axis in the orbital plane convention).
sensor_boresights_body = jnp.array([
    [ 1.0,  0.0, 0.0],   # +R face
    [-1.0,  0.0, 0.0],   # -R face
    # [ 0.0,  1.0, 0.0],   # +T face
    # [ 0.0, -1.0, 0.0],   # -T face
], dtype=jnp.float32)

# Alternative: single +R face only.
# sensor_boresights_body = jnp.array([[1.0, 0.0, 0.0]], dtype=jnp.float32)

# Alternative: two opposing faces (+R and -R).
# sensor_boresights_body = jnp.array([
#     [1.0, 0.0, 0.0],
#     [-1.0, 0.0, 0.0],
# ], dtype=jnp.float32)

half_angle_rad = jnp.deg2rad(15.0)   # ±15 deg half-angle per cone face (narrow = intermittent)

# ── Attitude parameters ────────────────────────────────────────────────────
# A 10 kg m² isotropic body with z-axis spin saturation at 0.5 rad/s.
# In 2D only ωz matters; ωx and ωy remain near zero throughout.
attitude_params = AttitudeParams(
    inertia_diag=jnp.array([10.0, 10.0, 5.0], dtype=jnp.float32),
    omega_max=jnp.array([0.0, 0.0, 0.5], dtype=jnp.float32),
)

# ── Attitude IC sampler: slow deterministic z-axis spin via FixedAttitude ──
# 0.1 deg/s × 10 s/step × 200 steps = 200° total — visible but slow.
# Replace with UniformBodyRates or UniformAttitude for randomized ICs.
_slow_spin_2d = FixedAttitude(
    quat_wxyz=(1.0, 0.0, 0.0, 0.0),
    omega_rad_s=(0.0, 0.0, float(jnp.deg2rad(0.1))),
)

# ── Proto config → layout → ConicalObservation ────────────────────────────
proto_cfg = ScenarioConfig(
    n_guards=1,
    n_bandits=1,
    epoch_mjd_utc=60067.0,
    reference_orbit=ReferenceOrbitState(
        position_eci=jnp.array([7000e3, 0.0, 0.0]),
        velocity_eci=jnp.array([0.0, 7.5e3, 0.0]),
    ),
    guard_components=(
        StateComponentKey.RT,
        StateComponentKey.MASS,
        StateComponentKey.ATTITUDE,
        StateComponentKey.BODY_RATES,
    ),
    bandit_components=(
        StateComponentKey.RT,
        StateComponentKey.ATTITUDE,
        StateComponentKey.BODY_RATES,
    ),
    guard_params=VehicleParamsSpec(dry_mass_kg=100.0, isp_s=220.0, max_thrust_n=5.0),
    bandit_params=VehicleParamsSpec(dry_mass_kg=50.0, isp_s=200.0, max_thrust_n=2.0),
    ic_sampler=ICSpec(
        guard_sampler=RelativeEllipse(
            radial_ellipse_m=1000.0, phase_rad=0.0, sigma_radial_ellipse_m=10.0,
            mass_sampler=ConstantMass(propellant_mass_kg=10.0),
            attitude_sampler=_slow_spin_2d,
        ),
        bandit_sampler=RelativeEllipse(
            radial_ellipse_m=1000.0, phase_rad=jnp.pi, sigma_radial_ellipse_m=10.0,
            attitude_sampler=_slow_spin_2d,
        ),
    ),
    truth_dynamics=DynamicsKey.HCW_RT,
    policy_dynamics=DynamicsKey.HCW_RT,
    action_frame=Frame.RT,
    guard_action_components=(
        ActionComponentKey.ATTITUDE_CONTROL,
        ActionComponentKey.IMPULSIVE_MANEUVER,
    ),
    bandit_action_components=(ActionComponentKey.IMPULSIVE_MANEUVER,),
    attitude_dynamics_key=AttitudeDynamicsKey.RIGID_BODY,
    guard_attitude_params=attitude_params,
    bandit_attitude_params=attitude_params,
    dt=10.0,
    max_horizon_s=2000.0,  # 200 steps: fast enough to see pointing dynamics
    seed=0,
)
proto_layout = OrbitalGymEnv(proto_cfg).layout

# Build the conical observation function using the resolved layout.
guard_obs_fn = ConicalObservation(
    layout=proto_layout,
    sensor_boresights_body=sensor_boresights_body,
    half_angle_rad=float(half_angle_rad),
    sigma=2.0,
)

# Symmetric sensor configuration on both sides — the bandit also has a
# body-fixed cone fan and must rotate to track the guard. This makes both
# sides' beliefs visibly fluctuate as their cones sweep.
bandit_obs_fn = ConicalObservation(
    layout=proto_layout,
    sensor_boresights_body=sensor_boresights_body,
    half_angle_rad=float(half_angle_rad),
    sigma=2.0,
)

# Final config with both observation functions slotted in.
cfg = ScenarioConfig(
    n_guards=1,
    n_bandits=1,
    epoch_mjd_utc=60067.0,
    reference_orbit=ReferenceOrbitState(
        position_eci=jnp.array([7000e3, 0.0, 0.0]),
        velocity_eci=jnp.array([0.0, 7.5e3, 0.0]),
    ),
    guard_components=(
        StateComponentKey.RT,
        StateComponentKey.MASS,
        StateComponentKey.ATTITUDE,
        StateComponentKey.BODY_RATES,
    ),
    bandit_components=(
        StateComponentKey.RT,
        StateComponentKey.ATTITUDE,
        StateComponentKey.BODY_RATES,
    ),
    guard_params=VehicleParamsSpec(dry_mass_kg=100.0, isp_s=220.0, max_thrust_n=5.0),
    bandit_params=VehicleParamsSpec(dry_mass_kg=50.0, isp_s=200.0, max_thrust_n=2.0),
    ic_sampler=ICSpec(
        guard_sampler=RelativeEllipse(
            radial_ellipse_m=1000.0, phase_rad=0.0, sigma_radial_ellipse_m=10.0,
            mass_sampler=ConstantMass(propellant_mass_kg=10.0),
            attitude_sampler=_slow_spin_2d,
        ),
        bandit_sampler=RelativeEllipse(
            radial_ellipse_m=1000.0, phase_rad=jnp.pi, sigma_radial_ellipse_m=10.0,
            attitude_sampler=_slow_spin_2d,
        ),
    ),
    truth_dynamics=DynamicsKey.HCW_RT,
    policy_dynamics=DynamicsKey.HCW_RT,
    action_frame=Frame.RT,
    guard_action_components=(
        ActionComponentKey.ATTITUDE_CONTROL,
        ActionComponentKey.IMPULSIVE_MANEUVER,
    ),
    bandit_action_components=(ActionComponentKey.IMPULSIVE_MANEUVER,),
    attitude_dynamics_key=AttitudeDynamicsKey.RIGID_BODY,
    guard_attitude_params=attitude_params,
    bandit_attitude_params=attitude_params,
    guard_observation_fn=guard_obs_fn,
    bandit_observation_fn=bandit_obs_fn,
    dt=10.0,
    max_horizon_s=2000.0,
    seed=0,
)

print(f"max_steps (derived):    {cfg.max_steps}")
print(f"guard action components: {[c.value for c in cfg.guard_action_components]}")
print(f"guard obs fn:            {type(cfg.guard_observation_fn).__name__}")
print(f"bandit obs fn:           {type(cfg.bandit_observation_fn).__name__}")
print(f"n_sensor_faces:          {sensor_boresights_body.shape[0]}")
print(f"half_angle_deg:          {float(jnp.rad2deg(half_angle_rad)):.1f}")
print(f"IC spin rate (deg/s):    {float(jnp.rad2deg(jnp.deg2rad(0.1))):.2f}")
print(f"  → 0.1 deg/s × 10s × 200 steps = {0.1*10*200:.0f} deg total")

max_steps (derived):    200
guard action components: ['attitude_control', 'impulsive_maneuver']
guard obs fn:            ConicalObservation
bandit obs fn:           ConicalObservation
n_sensor_faces:          2
half_angle_deg:          15.0
IC spin rate (deg/s):    0.10
  → 0.1 deg/s × 10s × 200 steps = 200 deg total


## 2. Construct the environment

`OrbitalGymEnv(cfg)` resolves all pluggables.  The flat state dim is larger
than in the plain 2D RT notebook because each vehicle now carries attitude
state.  Guard: RT(4) + MASS(1) + ATTITUDE(4) + BODY_RATES(3) + APPLIED_TORQUE(3) + APPLIED_DV(3) = 18.
Bandit: RT(4) + ATTITUDE(4) + BODY_RATES(3) + APPLIED_TORQUE(3) + APPLIED_DV(3) = 17.

Note: `APPLIED_TORQUE` and `APPLIED_DV` are transient bookkeeping fields —
they are written by action components and consumed by dynamics each step,
then zeroed.  They are included in the flat state for shape-stability inside
`jax.lax.scan`.


In [4]:
env = OrbitalGymEnv(cfg)
print(f"mean motion (rad/s):  {env.mean_motion:.3e}")
print(f"layout flat dim:      {env.layout.flat_dim}")
print(f"guard command fields: {list(env.guard_command_cls.zeros(1).__dataclass_fields__.keys())}")
print(f"bandit cmd fields:    {list(env.bandit_command_cls.zeros(1).__dataclass_fields__.keys())}")


mean motion (rad/s):  1.098e-03
layout flat dim:      35
guard command fields: ['torque', 'dv']
bandit cmd fields:    ['dv']


## 3. Policies — passive rotation only

Both sides use `ZeroControl`: no Δv, no torque commands.  Attitude evolves
*passively* from the body rate set by the IC sampler (0.1 °/s about the
N-axis in the 2D RT frame), so the cones sweep slowly and smoothly across
the rollout.  Free precession with zero torque means quaternion
integration is a pure RK4 over the kinematic equation
\(\dot{q}=\tfrac{1}{2}\,\Omega(\omega)\,q\), no controller dynamics
to fight.  This makes the visualization easy to read — every spacecraft
rotates at exactly the IC rate.

To turn this into an *active* pointing problem, the user would write a
heuristic that reads the bandit position from `env_state` (or the guard's
belief), compute an angle error, and command torque proportional to that
error — clipped to the reaction-wheel saturation `omega_max`.  We keep
the demonstration controller-free here so the cone-and-belief story is
visually unambiguous.

In [5]:
# Both sides emit zero commands; attitude rotates passively from the IC.
guard_policy = ZeroControl(n_vehicles=cfg.n_guards, command_cls=env.guard_command_cls)
bandit_policy = ZeroControl(n_vehicles=cfg.n_bandits, command_cls=env.bandit_command_cls)

def init_ps(config, env_state, key):
    del config, env_state, key
    return None

print("Guard policy:  ZeroControl (passive rotation from IC body rate)")
print("Bandit policy: ZeroControl (passive rotation from IC body rate)")


Guard policy:  ZeroControl (passive rotation from IC body rate)
Bandit policy: ZeroControl (passive rotation from IC body rate)


## 4. Manual rollout with attitude state

Because the heuristic reads the live env state, we run a Python-level loop
instead of `rollout()` (which uses `jax.lax.scan` and cannot close over
mutable Python state).  This is the same pattern used in
`tests/integration/test_attitude_scenario.py`.

The loop simultaneously:
- Steps the environment.
- Collects the observation channel (which carries the `visible` mask).
- Updates the EKF belief (for the belief rollout section below).


In [6]:
from orbitalgym.env.types import SideTrajectory, Trajectory
from orbitalgym.observations.types import flatten_observations

N_STEPS = cfg.max_steps

# ── EKF setup ─────────────────────────────────────────────────────────────
n_motion = float(ref_mean_motion(cfg.reference_orbit))
hcw_params = SimpleNamespace(mean_motion=n_motion)
d = env.layout.dynamics_state_dim  # 4 for RT

def per_vehicle_hcw_rt(x, u, dt):
    return hcw_rt_step(x[None, :], u[None, :], hcw_params, dt)[0]

ekf_updater = EKFBeliefUpdater(
    dynamics_fn=per_vehicle_hcw_rt,
    process_noise=jnp.eye(d, dtype=jnp.float32) * 0.01,
    dt=cfg.dt,
)
ekf_initializer = EKFFromTruthInitializer(
    layout=env.layout,
    variance_diag=jnp.ones(d, dtype=jnp.float32) * 100.0,
)

# ── Manual rollout loop ────────────────────────────────────────────────────
# IC body rates come from FixedAttitude via the attitude_sampler field —
# no manual state.replace() injection needed.
key = jax.random.PRNGKey(42)
k_reset, key = jax.random.split(key)
state, initial_outputs = env.reset(k_reset)


k_init, key = jax.random.split(key)
belief_guard = ekf_initializer(state, Side.GUARD, k_init)
k_init_b, key = jax.random.split(key)
belief_bandit = ekf_initializer(state, Side.BANDIT, k_init_b)

# Zero translational control for EKF prediction (no IMPULSIVE_MANEUVER dv).
zero_dv = jnp.zeros((cfg.n_guards, 2), dtype=jnp.float32)
zero_guard_cmd = env.guard_command_cls.zeros(cfg.n_guards)
zero_bandit_cmd = env.bandit_command_cls.zeros(cfg.n_bandits)
identity_actions = Actions(sides=BySide(guard=zero_guard_cmd, bandit=zero_bandit_cmd))

state_history          = []
obs_history            = []
reward_history         = []
guard_action_history   = []
bandit_action_history  = []
episode_done_history   = []
guard_obs_flat_history = []
bandit_obs_flat_history= []
belief_guard_history   = []
belief_bandit_history  = []

for step in range(N_STEPS):
    k_guard, k_bandit, k_env, k_obs, k_upd, key = jax.random.split(key, 6)

    # Guard command: heuristic torque (bandit: zero).
    guard_cmd, _ = guard_policy(None, None, k_guard, state.t)
    bandit_cmd = zero_bandit_cmd
    actions = Actions(sides=BySide(guard=guard_cmd, bandit=bandit_cmd))

    step_out = env.step(k_env, state, actions)
    state = step_out.state
    
    # Conical observation on the new state.
    obs_channels = guard_obs_fn(state, actions, Side.GUARD, cfg, k_obs, state.t)
    obs = obs_channels[0]  # single Observation namedtuple

    # Bandit conical observation on the new state.
    bandit_obs_channels = bandit_obs_fn(state, actions, Side.BANDIT, cfg, k_upd, state.t)

    # EKF update for both sides (predict + correct where visible).
    k_upd_g, k_upd_b = jax.random.split(k_upd)
    belief_guard = ekf_updater(belief_guard, obs_channels, zero_dv, Side.GUARD, k_upd_g)
    zero_dv_b = jnp.zeros((cfg.n_bandits, 2), dtype=jnp.float32)
    belief_bandit = ekf_updater(belief_bandit, bandit_obs_channels, zero_dv_b, Side.BANDIT, k_upd_b)

    state_history.append(state)
    obs_history.append(obs)
    reward_history.append(step_out.outputs.guard.reward)
    guard_action_history.append(guard_cmd)
    bandit_action_history.append(bandit_cmd)
    episode_done_history.append(step_out.episode_done)
    guard_obs_flat_history.append(flatten_observations(step_out.outputs.guard.obs))
    bandit_obs_flat_history.append(flatten_observations(step_out.outputs.bandit.obs))
    belief_guard_history.append(belief_guard)
    belief_bandit_history.append(belief_bandit)

# Stack into arrays for plotting.
rewards_arr = jnp.stack(reward_history, axis=0)
print(f"Episode return: {float(rewards_arr.sum()):.2f}")
print(f"Steps with at least one in-cone observation: "
      f"{sum(bool(o.visible.any()) for o in obs_history)} / {N_STEPS}")

# Sanity check: rotation is happening (IC spin from FixedAttitude sampler).
quat_t0 = state_history[0].guards.quat[0]
quat_tN = state_history[-1].guards.quat[0]
# Angle between two quaternions: 2 * arccos(|q0 · qN|)
dot = float(jnp.clip(jnp.abs(quat_t0 @ quat_tN), 0.0, 1.0))
total_angle_deg = float(jnp.rad2deg(2.0 * jnp.arccos(dot)))
print(f"Guard quaternion rotation t=0→t=N: {total_angle_deg:.1f} deg")
print(f"  (passive rotation: 0.1 deg/s × 10s/step × 200 steps = 200 deg total; quaternion angle wraps at 180 deg)")

# ── Build a Trajectory pytree from the loop history for RolloutScene ──────
traj_heuristic = Trajectory(
    env_state=jax.tree_util.tree_map(lambda *xs: jnp.stack(xs, axis=0), *state_history),
    sides=BySide(
        guard=SideTrajectory(
            obs=jnp.stack(guard_obs_flat_history, axis=0),
            action=jax.tree_util.tree_map(lambda *xs: jnp.stack(xs, axis=0), *guard_action_history),
            reward=jnp.stack(reward_history, axis=0),
            done=jnp.stack(episode_done_history, axis=0),
            policy_state=None,
        ),
        bandit=SideTrajectory(
            obs=jnp.stack(bandit_obs_flat_history, axis=0),
            action=jax.tree_util.tree_map(lambda *xs: jnp.stack(xs, axis=0), *bandit_action_history),
            reward=jnp.zeros(N_STEPS, dtype=jnp.float32),
            done=jnp.stack(episode_done_history, axis=0),
            policy_state=None,
        ),
    ),
    episode_done=jnp.stack(episode_done_history, axis=0),
    controlled_side=Side.GUARD,
)

# Stack belief histories: each entry is a Belief pytree; stack over time axis.
belief_history_stacked = BySide(
    guard=jax.tree_util.tree_map(lambda *xs: jnp.stack(xs, axis=0), *belief_guard_history),
    bandit=jax.tree_util.tree_map(lambda *xs: jnp.stack(xs, axis=0), *belief_bandit_history),
)

# Sanity: check belief covariance Frobenius norms at final step.
g_cov_norm = float(jnp.linalg.norm(belief_history_stacked.guard.cov[-1, 0, 1], ord="fro"))
b_cov_norm = float(jnp.linalg.norm(belief_history_stacked.bandit.cov[-1, 0, 1], ord="fro"))
print(f"Final guard belief cov norm (bandit block):  {g_cov_norm:.4f}")
print(f"Final bandit belief cov norm (guard block):  {b_cov_norm:.4f}")

Episode return: 0.00
Steps with at least one in-cone observation: 28 / 200
Guard quaternion rotation t=0→t=N: 161.0 deg
  (passive rotation: 0.1 deg/s × 10s/step × 200 steps = 200 deg total; quaternion angle wraps at 180 deg)


Final guard belief cov norm (bandit block):  369394.4375
Final bandit belief cov norm (guard block):  369394.4375


## 5. RT trajectory

The guard executes the spin-toward-target heuristic but commands zero Δv,
so the orbital trajectory is pure HCW dynamics (the 2:1 RT ellipse).
The bandit also has zero Δv.  The trajectory shows the relative motion
between the two spacecraft as the guard rotates to track the bandit.


In [7]:
# Stack guard RT positions from state history.
guard_rts = jnp.stack([s.guards.rt for s in state_history], axis=0)  # (T, 1, 4)
bandit_rts = jnp.stack([s.bandits.rt for s in state_history], axis=0)

fig, ax = plt.subplots(figsize=(8, 8))
plot_rt(guard_rts, ax=ax, label="guard")
plot_rt(bandit_rts, ax=ax, label="bandit")
ax.legend()
ax.set_title("R-T trajectory — attitude heuristic (zero Δv)")
plt.tight_layout()
plt.savefig("/tmp/rt_trajectory.png", dpi=80)
plt.show()
print("RT trajectory plotted.")


RT trajectory plotted.


/var/folders/66/25bqc2bn19bb84ddx70zwv9m0000gn/T/ipykernel_68797/2192983080.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Visibility heatmap — in-cone observation timing

This plot shows *when* the guard's cone sensor could see the bandit.
Each column is a time step; each row is an entity index in the observation
channel (index 0 = guard itself, index 1 = bandit for a 1v1 scenario).
White cells = in cone; black cells = out of cone.

The heatmap should **not** be a flat grey square — if it is, the cone is
always seeing or always missing the target, suggesting either a geometry
issue (guard never rotates) or a cone half-angle that is too wide/narrow.
With the four-face configuration and the spin-toward-target heuristic you
should see intermittent patterns as the guard rotates and the bandit drifts
around its relative orbit.

The guard's own row (index 0) is always False — the opposing-side mask in
`ConicalObservation` prevents self-observation.


In [8]:
# visible: (n_guards=1, n_total=2) boolean per step.
visibility = jnp.stack([o.visible for o in obs_history], axis=0)  # (T, 1, 2)
# Guard view (index 0), all entity slots across time.
vis_guard = visibility[:, 0, :]  # (T, 2)

fig, ax = plt.subplots(figsize=(10, 3))
ax.imshow(vis_guard.T, aspect="auto", origin="lower",
          cmap="Greys", interpolation="nearest",
          vmin=0, vmax=1)
ax.set_xlabel("step")
ax.set_ylabel("entity index")
ax.set_yticks([0, 1])
ax.set_yticklabels(["guard (self)", "bandit"])
ax.set_title("In-cone visibility over time (guard view)  — white=visible, black=not")
fig.tight_layout()
plt.savefig("/tmp/visibility_heatmap.png", dpi=80)
plt.show()

n_visible = int(vis_guard[:, 1].sum())
print(f"Steps bandit was in-cone: {n_visible} / {N_STEPS}  ({100*n_visible/N_STEPS:.1f}%)")


/var/folders/66/25bqc2bn19bb84ddx70zwv9m0000gn/T/ipykernel_68797/2629600975.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Steps bandit was in-cone: 28 / 200  (14.0%)


## 7. Reward curve

The default reward function penalises distance from the reference orbit.
With zero Δv the guard stays on its HCW ellipse, so the reward oscillates
with the orbital period.  The attitude heuristic does not change the
translational trajectory, so this curve is identical to the plain 2D RT
baseline — the interesting change is in the *observability* curve above.


In [9]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(rewards_arr[:, 0] if rewards_arr.ndim > 1 else rewards_arr, lw=1.5, color="steelblue")
mean_r = float(rewards_arr.mean())
ax.axhline(mean_r, color="red", ls="--", lw=1, label=f"mean={mean_r:.2f}")
ax.set_xlabel("step")
ax.set_ylabel("reward")
ax.set_title("Guard reward per step")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/reward_curve.png", dpi=80)
plt.show()


/var/folders/66/25bqc2bn19bb84ddx70zwv9m0000gn/T/ipykernel_68797/1088148783.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Propellant mass

With zero translational Δv commands the propellant remains constant — the
flat line confirms that `ATTITUDE_CONTROL` does not consume propellant (only
`IMPULSIVE_MANEUVER` triggers the rocket equation).  To see depletion,
uncomment the random Δv commands in the rollout loop and rerun.


In [10]:
prop_mass = jnp.stack([s.guards.propellant_mass for s in state_history], axis=0)  # (T, 1)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(prop_mass[:, 0], color="darkorange")
ax.set_xlabel("step")
ax.set_ylabel("propellant mass (kg)")
ax.set_title("Guard propellant over time (flat = no translational maneuvers)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/propellant_mass.png", dpi=80)
plt.show()


/var/folders/66/25bqc2bn19bb84ddx70zwv9m0000gn/T/ipykernel_68797/2339265883.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Persistence — save and reload via HDF5

`save_run` / `load_run` work unchanged for attitude scenarios.  The flat
trajectory dict will now contain keys like `"env_state.guards.quat"` and
`"env_state.guards.omega"` in addition to the standard RT and mass keys.

Because this notebook uses a hand-rolled loop rather than `rollout()`, we
reconstruct a minimal `Trajectory`-compatible dict for persistence.


In [11]:
import tempfile, pathlib, jax

# Re-run a quick jax.lax.scan rollout (ZeroControl, no heuristic) for
# save_run compatibility — save_run expects a Trajectory pytree, not a
# hand-rolled list.  This is the standard pattern from workflow_2d_rt.
from orbitalgym.rollout import rollout as _rollout

zero_g = ZeroControl(n_vehicles=cfg.n_guards, command_cls=env.guard_command_cls)
zero_b = ZeroControl(n_vehicles=cfg.n_bandits, command_cls=env.bandit_command_cls)

traj_save = _rollout(
    env,
    BySide(guard=zero_g, bandit=zero_b),
    BySide(guard=lambda c, s, k: None, bandit=lambda c, s, k: None),
    jax.random.PRNGKey(99),
    n_steps=cfg.max_steps,
)

tmp = pathlib.Path(tempfile.mkdtemp())
out = tmp / "rollout_attitude_2d.h5"
save_run(out, cfg, traj_save)

loaded_cfg, loaded_traj = load_run(out)
print(f"file written:           {out}  ({out.stat().st_size:,} bytes)")
print(f"loaded config matches:  {loaded_cfg.n_guards == cfg.n_guards and loaded_cfg.dt == cfg.dt}")
attitude_keys = [k for k in sorted(loaded_traj.keys()) if "quat" in k or "omega" in k or "torque" in k]
print(f"attitude-related keys:  {attitude_keys}")
print(f"reward round-trips:     {bool(jnp.allclose(loaded_traj['sides.guard.reward'], traj_save.sides.guard.reward))}")


file written:           /var/folders/66/25bqc2bn19bb84ddx70zwv9m0000gn/T/tmp3xyvarzu/rollout_attitude_2d.h5  (116,271 bytes)
loaded config matches:  True
attitude-related keys:  ['env_state.bandits.applied_torque', 'env_state.bandits.omega', 'env_state.bandits.quat', 'env_state.guards.applied_torque', 'env_state.guards.omega', 'env_state.guards.quat', 'sides.guard.action.torque']
reward round-trips:     True


## 10. Animation — cones sweeping with attitude

`RolloutScene` with `show_sensor_cones=True` reads the guard's quaternion
at each frame and draws the body-fixed cone faces rotated into the world
frame.  With `mode="auto"` the scene selects a 2D (RT-plane) view because
both sides carry only the `RT` spatial component (no `RTN`).

`cone_length_m=200.0` scales the cone visualization to be visible at the
1 km orbit scale.  Adjust this to taste.

The animation is saved as an MP4 using ffmpeg via `save_animation`.


Save the animation in two formats: MP4 (smaller, requires ffmpeg) and GIF
(larger but universally embeddable in markdown / web previews / Slack /
GitHub READMEs without a player). The `save_animation` helper dispatches
on file extension — `.mp4` uses ffmpeg, `.gif` uses matplotlib's bundled
PillowWriter.

In [12]:
# Build scene from the heuristic rollout (traj_heuristic from the rollout
# cell above). The IC body rates come from FixedAttitude via attitude_sampler,
# so no manual omega injection is needed — the trajectory already has the
# correct initial spin baked in from env.reset().
scene = RolloutScene(
    traj=traj_heuristic,
    cfg=cfg,
    dt=cfg.dt,
    show_cubes=True,
    show_trail=True,
    show_sensor_cones=True,
    show_thrust=False,
    show_sensor_range=False,
    show_belief=True,
    belief_history=belief_history_stacked,
    title_prefix="Attitude 2D RT  ",
)
print(f"resolved mode:  {scene.resolved_mode}")
print(f"axis limit (m): {scene.axis_limit_m:.0f}")
print(f"n_frames:       {scene.n_frames}")
print(f"eff cone length (m): {2.5 * scene.axis_limit_m:.0f}  (2.5 × axis_limit)")

import shutil
out_dir = pathlib.Path("../outputs")
out_dir.mkdir(parents=True, exist_ok=True)

# GIF always works (PillowWriter is bundled with matplotlib)
gif_path = out_dir / "workflow_2d_rt_attitude.gif"
save_animation(scene, str(gif_path), fps=15, dpi=90)
print(f"saved {gif_path}  ({gif_path.stat().st_size:,} bytes)")

# MP4 is smaller and smoother but needs ffmpeg on PATH
if shutil.which("ffmpeg") is not None:
    mp4_path = out_dir / "workflow_2d_rt_attitude.mp4"
    save_animation(scene, str(mp4_path), fps=15, dpi=90)
    print(f"saved {mp4_path}  ({mp4_path.stat().st_size:,} bytes)")
else:
    print("ffmpeg not available — skipping MP4 export")

# In-notebook playback: prefer MP4 (smaller, smoother) when ffmpeg produced
# one, fall back to the GIF otherwise. ``embed=False`` keeps the notebook
# small by referencing the file path instead of base64-encoding the bytes.
from IPython.display import Image, Video, display

if shutil.which("ffmpeg") is not None and mp4_path.exists():
    display(Video(str(mp4_path), embed=False))
else:
    display(Image(str(gif_path)))


resolved mode:  2d
axis limit (m): 2224
n_frames:       200
eff cone length (m): 5561  (2.5 × axis_limit)


saved ../outputs/workflow_2d_rt_attitude.gif  (2,538,706 bytes)


saved ../outputs/workflow_2d_rt_attitude.mp4  (261,576 bytes)
